In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split, ConcatDataset, Dataset
from PIL import Image

In [2]:

!kaggle datasets download -d abdallahalidev/plantvillage-dataset
!unzip -q plantvillage-dataset.zip
!kaggle datasets download -d prasunroy/natural-images
!unzip -q natural-images.zip


Dataset URL: https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset
License(s): CC-BY-NC-SA-4.0
100% 2.04G/2.04G [00:09<00:00, 220MB/s]

Dataset URL: https://www.kaggle.com/datasets/prasunroy/natural-images
License(s): CC-BY-NC-SA-4.0
100% 342M/342M [00:02<00:00, 152MB/s]



In [3]:
plant_dir = "plantvillage dataset/color"
natural_dir = "natural_images"
print(os.listdir(plant_dir)[:5])
print(os.listdir(natural_dir))

['Apple___healthy', 'Tomato___Spider_mites Two-spotted_spider_mite', 'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Potato___Late_blight', 'Tomato___Target_Spot']
['fruit', 'cat', 'person', 'airplane', 'motorbike', 'flower', 'dog', 'car']


In [4]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [5]:
plant_dataset = datasets.ImageFolder(plant_dir)
natural_dataset = datasets.ImageFolder(natural_dir)
print(len(plant_dataset.classes), len(plant_dataset))
print(len(natural_dataset.classes), len(natural_dataset))


38 54305
8 6899


In [6]:
all_classes = plant_dataset.classes + natural_dataset.classes
num_classes = len(all_classes)
print(num_classes)

46


In [7]:
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, subset, transform=None, label_offset=0):
        self.subset = subset
        self.transform = transform
        self.label_offset = label_offset
    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y + self.label_offset
    def __len__(self):
        return len(self.subset)


In [8]:
def split_dataset(ds, ratio=0.8):
    n = len(ds)
    train = int(ratio * n)
    return random_split(ds, [train, n - train])

plant_train, plant_val = split_dataset(plant_dataset)
natural_train, natural_val = split_dataset(natural_dataset)
natural_offset = len(plant_dataset.classes)

train_dataset = ConcatDataset([
    CustomDataset(plant_train, train_transform, label_offset=0),
    CustomDataset(natural_train, train_transform, label_offset=natural_offset),
])
val_dataset = ConcatDataset([
    CustomDataset(plant_val, val_transform, label_offset=0),
    CustomDataset(natural_val, val_transform, label_offset=natural_offset),
])
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
print(len(train_dataset), len(val_dataset))

48963 12241


In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.mobilenet_v2(weights="IMAGENET1K_V1")
for param in model.parameters():
    param.requires_grad = False
model.classifier[1] = nn.Linear(model.last_channel, num_classes)
model = model.to(device)


Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 96.0MB/s]


In [10]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

In [11]:
num_epochs = 10
best_acc = 0.0
val_size = len(val_dataset)
for epoch in range(num_epochs):
    model.train()
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
    model.eval()
    val_correct = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            preds = torch.argmax(outputs, 1)
            val_correct += (preds == labels).sum().item()
    val_acc = val_correct / val_size
    print(f"Epoch {epoch+1}/{num_epochs} - Val Accuracy: {val_acc:.4f}")
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "combined_model.pth")

Epoch 1/10 - Val Accuracy: 0.9466
Epoch 2/10 - Val Accuracy: 0.9565
Epoch 3/10 - Val Accuracy: 0.9577
Epoch 4/10 - Val Accuracy: 0.9605
Epoch 5/10 - Val Accuracy: 0.9648
Epoch 6/10 - Val Accuracy: 0.9553
Epoch 7/10 - Val Accuracy: 0.9656
Epoch 8/10 - Val Accuracy: 0.9633
Epoch 9/10 - Val Accuracy: 0.9558
Epoch 10/10 - Val Accuracy: 0.9693


In [12]:
def predict(image_path):
    eval_model = models.mobilenet_v2(weights=None)
    eval_model.classifier[1] = nn.Linear(eval_model.last_channel, num_classes)
    eval_model.load_state_dict(torch.load("combined_model.pth", map_location=device))
    eval_model = eval_model.to(device)
    eval_model.eval()
    img = Image.open(image_path).convert("RGB")
    img = val_transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        pred = torch.argmax(eval_model(img), 1)
    return all_classes[pred.item()]


In [21]:
from google.colab import files
uploaded = files.upload()
image_name = list(uploaded.keys())[0]
print(predict(image_name))

Saving تنزيل (1).jfif to تنزيل (1).jfif
Apple___Black_rot


In [22]:
from google.colab import files
files.download("combined_model.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>